# LangChain Tool Stream with Gemini API

이 노트북은 Google Gemini API를 사용하여 LangChain Tools와 스트림 출력을 구현하는 방법을 보여줍니다.

## 주요 학습 내용:
- Gemini API를 사용한 기본 채팅
- 시간 조회 및 주식 조회 도구 구현
- 실시간 스트림 출력
- 도구 호출과 스트림 출력의 조합
- 파편화된 도구 호출 청크 처리

## 1. 환경 설정 및 모델 초기화

In [1]:
import os
import google.generativeai as genai
import pytz
import yfinance as yf
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from datetime import datetime
from pydantic import BaseModel, Field

# yfinance 설치 확인 및 안내
try:
    import yfinance as yf
    print("✅ yfinance 패키지가 설치되어 있습니다.")
except ImportError:
    print("⚠️ yfinance 패키지가 설치되지 않았습니다.")
    print("📝 설치 명령어: pip install yfinance")
    print("🔧 또는 UV 사용: uv pip install yfinance")

✅ yfinance 패키지가 설치되어 있습니다.


In [2]:
# 환경변수 로드
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("환경변수 GOOGLE_API_KEY가 설정되지 않았습니다.")
genai.configure(api_key=api_key)

print("✅ 환경변수가 성공적으로 로드되었습니다.")

✅ 환경변수가 성공적으로 로드되었습니다.


In [3]:
# Gemini 모델 초기화 - API 할당량 절약을 위한 설정
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",  # gemini-2.0-flash 대신 1.5-flash 사용 (더 안정적)
    temperature=0.7,
    max_retries=2,  # 재시도 횟수 제한
)

print("✅ Gemini 모델이 초기화되었습니다.")
print("⚠️ API 할당량 절약을 위해 gemini-1.5-flash 모델을 사용합니다.")

✅ Gemini 모델이 초기화되었습니다.
⚠️ API 할당량 절약을 위해 gemini-1.5-flash 모델을 사용합니다.


## 2. 기본 채팅 테스트

In [4]:
# 기본 채팅 테스트
response = llm.invoke([HumanMessage("잘 지냈어?")])
print(f"🤖 Gemini 응답: {response.content}")
response

🤖 Gemini 응답: 네, 잘 지냈어요.  혹시 제가 도와드릴 일이 있나요?


AIMessage(content='네, 잘 지냈어요.  혹시 제가 도와드릴 일이 있나요?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--d2dac53c-246c-4a49-9505-2f6f4785994f-0')

## 3. 도구 정의 - 시간 조회

In [5]:
@tool  # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    try:
        tz = pytz.timezone(timezone)
        now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        location_and_local_time = f'{timezone} ({location}) 현재시각 {now}'
        print(f"🕐 시간 조회: {location_and_local_time}")
        return location_and_local_time
    except Exception as e:
        error_msg = f"시간 조회 실패: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg

print("✅ 시간 조회 도구가 정의되었습니다.")

✅ 시간 조회 도구가 정의되었습니다.


## 4. Pydantic 모델 정의

In [6]:
class StockHistoryInput(BaseModel):
    """주식 조회를 위한 입력 모델"""
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL, TSLA)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

print("✅ StockHistoryInput 모델이 정의되었습니다.")

✅ StockHistoryInput 모델이 정의되었습니다.


## 5. 도구 정의 - 주식 조회

In [7]:
@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    try:
        # import yfinance as yf
        
        ticker = stock_history_input.ticker
        period = stock_history_input.period
        
        print(f"📈 주식 조회: {ticker} ({period})")
        
        stock = yf.Ticker(ticker)
        history = stock.history(period=period)
        
        if history.empty:
            return f"❌ {ticker} 주식 데이터를 찾을 수 없습니다."
        
        # 최근 5개 데이터만 표시 (출력 크기 제한)
        recent_data = history.tail(5)
        history_md = recent_data.to_markdown()
        return f"📊 {ticker} 주식 데이터 (최근 5일):\n{history_md}"
        
    except ImportError:
        return "❌ yfinance 패키지가 설치되지 않았습니다. 'uv pip install yfinance'로 설치해주세요."
    except Exception as e:
        error_msg = f"주식 데이터 조회 실패: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg

print("✅ 주식 조회 도구가 정의되었습니다.")

✅ 주식 조회 도구가 정의되었습니다.


## 6. 도구 설정 및 바인딩

In [8]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time, get_yf_stock_history]
tool_dict = {
    "get_current_time": get_current_time, 
    "get_yf_stock_history": get_yf_stock_history
}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

print("✅ 도구들이 모델에 바인딩되었습니다.")
print(f"📋 사용 가능한 도구: {[tool.name for tool in tools]}")

✅ 도구들이 모델에 바인딩되었습니다.
📋 사용 가능한 도구: ['get_current_time', 'get_yf_stock_history']


## 7. 시간 조회 테스트

In [9]:
# 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

print("💬 사용자 질문: 부산은 지금 몇시야?")

# llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# 생성된 llm 답변 출력
print(f"🤖 AI 응답: {response.content}")
print(f"📋 메시지 수: {len(messages)}")
print(messages)

💬 사용자 질문: 부산은 지금 몇시야?
🤖 AI 응답: 
📋 메시지 수: 3
[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"location": "Busan", "timezone": "Asia/Seoul"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--4b49e63e-b9de-4825-8649-e38508d1dd83-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': '3031a98a-6820-42bd-a971-be56a3cf2049', 'type': 'tool_call'}])]


In [10]:
if hasattr(response, 'tool_calls') and response.tool_calls:
    print(f"🔧 호출된 도구 수: {len(response.tool_calls)}")
    
    for tool_call in response.tool_calls:
        selected_tool = tool_dict[tool_call["name"]]  # tool_dict를 사용하여 도구 함수를 선택
        print(f"🛠️ 도구 호출: {tool_call['name']}")
        print(f"📥 전달된 인자: {tool_call['args']}")  # 도구 호출 시 전달된 인자 출력
        
        tool_msg = selected_tool.invoke(tool_call)  # 도구 함수를 호출하여 결과를 반환
        messages.append(tool_msg)

messages

🔧 호출된 도구 수: 1
🛠️ 도구 호출: get_current_time
📥 전달된 인자: {'location': 'Busan', 'timezone': 'Asia/Seoul'}
🕐 시간 조회: Asia/Seoul (Busan) 현재시각 2025-09-04 14:30:43


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"location": "Busan", "timezone": "Asia/Seoul"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--4b49e63e-b9de-4825-8649-e38508d1dd83-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': '3031a98a-6820-42bd-a971-be56a3cf2049', 'type': 'tool_call'}]),
 ToolMessage(content='Asia/Seoul (Busan) 현재시각 2025-09-04 14:30:43', name='get_current_time', tool_call_id='3031a98a-6820-42bd-a971-be56a3cf2049')]

In [11]:
# 최종 답변 생성
final_response = llm_with_tools.invoke(messages)
print(f"🎯 최종 답변: {final_response.content}")
final_response

🎯 최종 답변: 부산은 현재 2025년 9월 4일 14시 30분 43초 입니다.


AIMessage(content='부산은 현재 2025년 9월 4일 14시 30분 43초 입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--a08612f9-31aa-46d0-a688-77cf95b4946a-0')

## 8. 주식 조회 테스트

In [12]:
# 새로운 대화로 주식 조회 테스트
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

print("💬 사용자 질문: 테슬라는 한달 전에 비해 주가가 올랐나 내렸나?")

response = llm_with_tools.invoke(messages)
print(f"🤖 AI 응답: {response.content}")
print(response)
messages.append(response)

💬 사용자 질문: 테슬라는 한달 전에 비해 주가가 올랐나 내렸나?
🤖 AI 응답: 
content='' additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"period": "1mo", "ticker": "TSLA"}}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []} id='run--60db0181-b8db-4238-9620-2b1b0335ff0d-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}, 'id': '27c39be2-d645-482a-a461-f47a071db315', 'type': 'tool_call'}]


In [13]:
if hasattr(response, 'tool_calls') and response.tool_calls:
    print(f"🔧 호출된 도구 수: {len(response.tool_calls)}")
    
    for tool_call in response.tool_calls:
        selected_tool = tool_dict[tool_call["name"]]
        print(f"🛠️ 도구 호출: {tool_call['name']}")
        print(f"📥 전달된 인자: {tool_call['args']}")
        
        tool_msg = selected_tool.invoke(tool_call)
        messages.append(tool_msg)
        print(f"📤 도구 실행 결과 (일부): {tool_msg.content[:200]}...")
        print(tool_msg)

🔧 호출된 도구 수: 1
🛠️ 도구 호출: get_yf_stock_history
📥 전달된 인자: {'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}
📈 주식 조회: TSLA (1mo)
📤 도구 실행 결과 (일부): 📊 TSLA 주식 데이터 (최근 5일):
| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |
|:--------------------------|-------:|-------:|-------:|--------:...
content='📊 TSLA 주식 데이터 (최근 5일):\n| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2025-08-27 00:00:00-04:00 | 351.94 | 355.39 | 349.16 |  349.6  | 6.5519e+07  |           0 |              0 |\n| 2025-08-28 00:00:00-04:00 | 350.91 | 353.55 | 340.26 |  345.98 | 6.79032e+07 |           0 |              0 |\n| 2025-08-29 00:00:00-04:00 | 347.23 | 348.75 | 331.7  |  333.87 | 8.11457e+07 |           0 |              0 |\n| 2025-09-02 00:00:00-04:00 | 328.23 | 333.33

In [14]:
# 최종 답변
final_response = llm_with_tools.invoke(messages)
print(f"🎯 최종 답변: {final_response.content}")
final_response

🎯 최종 답변: 제공된 데이터는 최근 5일 동안의 테슬라 주가를 보여줍니다.  한 달 전의 데이터는 없으므로 한 달 전과 비교하여 주가가 상승했는지 하락했는지 판단할 수 없습니다.  더 긴 기간의 데이터를 조회해야 정확한 비교가 가능합니다.


AIMessage(content='제공된 데이터는 최근 5일 동안의 테슬라 주가를 보여줍니다.  한 달 전의 데이터는 없으므로 한 달 전과 비교하여 주가가 상승했는지 하락했는지 판단할 수 없습니다.  더 긴 기간의 데이터를 조회해야 정확한 비교가 가능합니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--f0600dcb-de7c-44c1-adfe-18de33f9cabc-0')

## 9. 기본 스트림 출력

In [15]:
print("💬 사용자 질문: 잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")
print("🔄 스트림 응답:")
print("-" * 50)

# 출력 길이 제한을 위한 카운터
chunk_count = 0
max_chunks = 50  # 최대 50개 청크만 표시

try:
    for chunk in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 간단히 이야기해줘.")]):
        if chunk_count >= max_chunks:
            print("\n⚠️ 출력이 너무 길어서 일부만 표시됩니다...")
            break
        print(chunk.content, end='!', flush=True)
        chunk_count += 1
except Exception as e:
    print(f"❌ API 할당량 초과 또는 네트워크 오류: {str(e)}")
    print("💡 나중에 다시 시도하거나, API 키 할당량을 확인해주세요.")

print("\n" + "-" * 50)

💬 사용자 질문: 잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.
🔄 스트림 응답:
--------------------------------------------------
네!, 잘 지냈습니다. 한국 사회의 문제점은 다양하고 복!잡하게 얽혀있지만, 몇 가지 주요한 것들을! 간략하게 이야기해보겠습니다.

* **극심한 경쟁 사회:** 교육, 취업, 주택 등 모든 영역에서! 과도한 경쟁이 만연하여 개인의 정신적, 육체적 건강을 해치고 사회적 불안을 증!폭시킵니다.  성공에 대한 압박이 크고, 실패에 대한 관용이 부족한 편입니다.

* **양극화 심화:** 소득 불균형이 심화되면!서 계층 이동의 어려움이 커지고, 사회적 불신과 갈등이 증가하고 있습니다.  부의 집중 현상 또한 심각한 문제입니다.

* **저출산 및! 고령화:** 낮은 출산율과 빠른 고령화는 사회 시스템 유지에 큰 부담을 주고 있으며, 미래 성장 동력 약화의 원인이 되고 있습니다.  젊은 세대의 삶의 질 저하와 미래에 대한! 불안감이 이 문제의 주요 원인 중 하나입니다.

* **청년 세대의 어려움:** 취업난, 주택 문제, 고용 불안정 등 청년 세대가 직면한 어려움은 심각하며, 이는 사회 전반의 활!력 저하로 이어집니다.

* **부동산 문제:**  높은 집값과 과열된 부동산 시장은 사회적 불평등을 심화시키고, 청년 세대의 삶에 큰 부담을 주고 있습니다.  투기 수!요와 제도적 미비점이 문제의 복합적인 원인으로 지적됩니다.


이 외에도,  장시간 노동,  젠더 불평등,  환경 문제 등 다양한 문제들이 한국 사회의 발전을 저해하고 있습니다.  이러한 문제!들은 서로 복잡하게 얽혀있어,  해결책을 찾기 위해서는 다각적인 접근과 사회 구성원들의 노력이 필요합니다.  이것은 단순히 나열된 문제점 이상으로,  한국 사회가 안고 있는 심각한 과제들!입니다.
!
--------------------------------------------------


## 10. 도구 사용 스트림 출력 - 청크 분석

In [16]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

print("💬 사용자 질문: 부산은 지금 몇시야?")
print("🔄 도구 호출 스트림 분석:")

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print(f"청크 타입: {type(chunk).__name__}")
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print(f"내용: '{gathered.content}', 도구 호출: {len(gathered.tool_calls) if gathered.tool_calls else 0}개")
    print("-" * 30)

messages.append(gathered)

💬 사용자 질문: 부산은 지금 몇시야?
🔄 도구 호출 스트림 분석:
청크 타입: AIMessageChunk
내용: '', 도구 호출: 1개
------------------------------


In [17]:
print(f"🔧 통합된 응답:")
print(gathered)
print(f"\n📋 도구 호출 정보: {gathered.tool_calls if gathered.tool_calls else '없음'}")

🔧 통합된 응답:
content='' additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []} id='run--079a00c6-8db3-42c8-8278-02e385e8121c' tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': '134a3b53-a5c2-40d7-8f05-f1c60b99194b', 'type': 'tool_call'}] tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}', 'id': '134a3b53-a5c2-40d7-8f05-f1c60b99194b', 'index': None, 'type': 'tool_call_chunk'}]

📋 도구 호출 정보: [{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': '134a3b53-a5c2-40d7-8f05-f1c60b99194b', 'type': 'tool_call'}]


In [18]:
# 도구 실행
if hasattr(gathered, 'tool_calls') and gathered.tool_calls:
    for tool_call in gathered.tool_calls:
        selected_tool = tool_dict[tool_call["name"]]  # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
        print(f"🛠️ 도구 호출: {tool_call['name']}")
        print(f"📥 전달된 인자: {tool_call['args']}")  # 도구 호출 시 전달된 인자 출력
        
        tool_msg = selected_tool.invoke(tool_call)  # 도구 함수를 호출하여 결과를 반환
        messages.append(tool_msg)

print(f"📋 총 메시지 수: {len(messages)}")
messages

🛠️ 도구 호출: get_current_time
📥 전달된 인자: {'timezone': 'Asia/Seoul', 'location': '부산'}
🕐 시간 조회: Asia/Seoul (부산) 현재시각 2025-09-04 14:30:50
📋 총 메시지 수: 4


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--079a00c6-8db3-42c8-8278-02e385e8121c', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': '134a3b53-a5c2-40d7-8f05-f1c60b99194b', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}', 'id': '134a3b53-a5c2-40d7-8f05-f1c60b99194b', 'index': None, 'type': 'tool_call_chunk'}]),
 ToolMessage(content='Asia/Seoul (부산) 현재시각 2025-09-04 14:30:50', name='get_current_time', tool_call_id='134a3b53-a5c2-40d

## 11. 최종 답변 스트림

In [19]:
print("🎯 최종 답변 스트림:")
print("-" * 50)

for chunk in llm_with_tools.stream(messages):
    print(chunk.content, end='', flush=True)

print("\n" + "-" * 50)

🎯 최종 답변 스트림:
--------------------------------------------------
부산은 현재 2025년 9월 4일 14시 30분 50초 입니다.

--------------------------------------------------


## 12. 사용법 및 정보

In [20]:
print("🔧 스트림 출력의 특징:")
print("  - 실시간으로 응답을 받을 수 있음")
print("  - 긴 응답의 경우 사용자 경험 향상")
print("  - 도구 호출 시 청크들이 파편화되어 전달됨")
print("  - 파편화된 청크를 하나로 합쳐야 함")

print("\n🛠️ 필요한 패키지:")
print("  - uv pip install yfinance (주식 데이터용)")
print("  - uv pip install pytz (시간대 처리용)")
print("  - uv pip install pydantic (데이터 검증용)")

print("\n⚠️ 주의사항:")
print("  - Gemini API 무료 티어: 하루 200회 요청 제한")
print("  - API 할당량 초과시 24시간 대기 또는 유료 플랜 업그레이드")
print("  - 스트림 모드에서는 도구 호출이 여러 청크로 나뉘어 전달")
print("  - gathered 객체로 청크를 누적해야 완전한 도구 호출 정보 확보")
print("  - 인터넷 연결이 필요한 기능들 (yfinance)")

print("\n🚨 현재 발생한 문제들:")
print("  1. API 할당량 초과 (429 오류)")
print("  2. yfinance 패키지 미설치")
print("  3. 출력 크기가 너무 커서 일부 표시 제한")

print("\n💡 해결 방법:")
print("  1. 24시간 후 다시 시도하거나 유료 플랜 사용")
print("  2. 터미널에서 'uv pip install yfinance' 실행")
print("  3. gemini-1.5-flash 모델 사용으로 안정성 개선")

print("\n💡 핵심 학습 포인트:")
print("  ✅ Gemini API를 사용한 실시간 스트림 응답")
print("  ✅ 도구 호출과 스트림 출력의 조합")
print("  ✅ 파편화된 도구 호출 청크 처리 방법")
print("  ✅ Pydantic 모델을 활용한 구조화된 입력")
print("  ✅ 복합 워크플로 (도구 실행 → 스트림 응답)")
print("  ✅ API 할당량 관리 및 오류 처리")

🔧 스트림 출력의 특징:
  - 실시간으로 응답을 받을 수 있음
  - 긴 응답의 경우 사용자 경험 향상
  - 도구 호출 시 청크들이 파편화되어 전달됨
  - 파편화된 청크를 하나로 합쳐야 함

🛠️ 필요한 패키지:
  - uv pip install yfinance (주식 데이터용)
  - uv pip install pytz (시간대 처리용)
  - uv pip install pydantic (데이터 검증용)

⚠️ 주의사항:
  - Gemini API 무료 티어: 하루 200회 요청 제한
  - API 할당량 초과시 24시간 대기 또는 유료 플랜 업그레이드
  - 스트림 모드에서는 도구 호출이 여러 청크로 나뉘어 전달
  - gathered 객체로 청크를 누적해야 완전한 도구 호출 정보 확보
  - 인터넷 연결이 필요한 기능들 (yfinance)

🚨 현재 발생한 문제들:
  1. API 할당량 초과 (429 오류)
  2. yfinance 패키지 미설치
  3. 출력 크기가 너무 커서 일부 표시 제한

💡 해결 방법:
  1. 24시간 후 다시 시도하거나 유료 플랜 사용
  2. 터미널에서 'uv pip install yfinance' 실행
  3. gemini-1.5-flash 모델 사용으로 안정성 개선

💡 핵심 학습 포인트:
  ✅ Gemini API를 사용한 실시간 스트림 응답
  ✅ 도구 호출과 스트림 출력의 조합
  ✅ 파편화된 도구 호출 청크 처리 방법
  ✅ Pydantic 모델을 활용한 구조화된 입력
  ✅ 복합 워크플로 (도구 실행 → 스트림 응답)
  ✅ API 할당량 관리 및 오류 처리


## 13. 요약 및 확장 가능성

In [21]:
print("🎉 LangChain Tool Stream with Gemini API 완료!")
print("=" * 60)

print("\n📚 구현한 주요 기능:")
features = [
    "- Gemini API를 사용한 기본 채팅 및 스트림 출력",
    "- 시간 조회 도구 (pytz 활용)",
    "- 주식 조회 도구 (yfinance 활용)",
    "- Pydantic 모델을 통한 입력 검증",
    "- 파편화된 도구 호출 청크 처리",
    "- 복합 워크플로 (도구 → 스트림)"
]
for feature in features:
    print(f"  {feature}")

print("\n🚀 확장 가능한 방향:")
extensions = [
    "- 더 많은 금융 데이터 소스 통합",
    "- 실시간 알림 시스템",
    "- 웹 인터페이스 (Streamlit/Gradio)",
    "- 대화 히스토리 저장 및 관리",
    "- 다국어 지원 확장"
]
for ext in extensions:
    print(f"  {ext}")

print("\n✅ 모든 테스트가 완료되었습니다!")

🎉 LangChain Tool Stream with Gemini API 완료!

📚 구현한 주요 기능:
  - Gemini API를 사용한 기본 채팅 및 스트림 출력
  - 시간 조회 도구 (pytz 활용)
  - 주식 조회 도구 (yfinance 활용)
  - Pydantic 모델을 통한 입력 검증
  - 파편화된 도구 호출 청크 처리
  - 복합 워크플로 (도구 → 스트림)

🚀 확장 가능한 방향:
  - 더 많은 금융 데이터 소스 통합
  - 실시간 알림 시스템
  - 웹 인터페이스 (Streamlit/Gradio)
  - 대화 히스토리 저장 및 관리
  - 다국어 지원 확장

✅ 모든 테스트가 완료되었습니다!
